# AlphaDent YOLO26 Segmentation Pipeline (End-to-End)

This notebook contains the complete pipeline for downloading the AlphaDent dataset, training, validating, and performing inference using **YOLO26 segment models**.

### Models Supported:
- `yolo26n-seg.pt` (Nano)
- `yolo26s-seg.pt` (Small)
- `yolo26m-seg.pt` (Medium)
- `yolo26l-seg.pt` (Large)
- `yolo26x-seg.pt` (Extra Large)

Developed for running on **Kaggle**.

## 1. Clone Repository & Setup Working Directory

In [ ]:
# Clone repo if not running inside it already
import os
if not os.path.exists('/kaggle/working/AlphaDentYolov26'):
    print("Cloning AlphaDentYolov26 repository...")
    !git clone https://github.com/ShMazumder/AlphaDentYolov26 /kaggle/working/AlphaDentYolov26/
else:
    print("Repository already exists.")

# Change directory to the repository root
os.chdir('/kaggle/working/AlphaDentYolov26/')
print(f"Current working directory: {os.getcwd()}")

# Install dependencies
print("Installing dependencies...")
!pip install -q -r requirements.txt
!pip install -q matplotlib opencv-python

## 2. Configuration

Configure model selection and training hyperparameters here.

In [ ]:
# Choose model: 'yolo26n-seg.pt', 'yolo26s-seg.pt', 'yolo26m-seg.pt', 'yolo26l-seg.pt', 'yolo26x-seg.pt'
MODEL_NAME = 'yolo26x-seg.pt'

# Dataset configuration file path
DATA_CONFIG = './AlphaDent/yolo26_seg_train.yaml'

# Training Settings
BATCH_SIZE = 8       # reduce if CUDA out of memory
EPOCHS = 50          # recommended: 100
IMAGE_SIZE = 640     # input size (640 or 960)

# Dynamic project naming to prevent overwriting results between different models/runs
MODEL_BASE = MODEL_NAME.replace('-seg.pt', '').replace('.pt', '')
PROJECT_NAME = f'yolo_seg_{MODEL_BASE}_proj_{IMAGE_SIZE}'

# Validation & Inference Settings
CONF_THRESHOLD = 0.001
IOU_THRESHOLD = 0.5

print(f"Configured model: {MODEL_NAME}")
print(f"Dataset config: {DATA_CONFIG}")
print(f"Project directory: {PROJECT_NAME}")
print(f"Batch size: {BATCH_SIZE}, Epochs: {EPOCHS}, Image size: {IMAGE_SIZE}")

## 3. Dataset Download & Unpacking

Download the dataset from Zenodo and extract it to the correct path.

In [ ]:
import zipfile
import shutil

dataset_dir = "/kaggle/working/AlphaDentYolov26/AlphaDent"
zip_path = "/kaggle/working/dataset.zip"

if not os.path.exists(dataset_dir) or not os.listdir(dataset_dir):
    print("Downloading AlphaDent dataset from Zenodo...")
    !wget -q --show-progress "https://zenodo.org/records/16582489/files/AlphaDent.zip?download=1" -O {zip_path}
    
    print("Extracting dataset...")
    os.makedirs(dataset_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dataset_dir)
    
    # Remove zip to save disk space
    if os.path.exists(zip_path):
        os.remove(zip_path)
        
    # Check if nested folder exists after extraction and move files up if needed
    nested_dir = os.path.join(dataset_dir, "AlphaDent")
    if os.path.exists(nested_dir) and os.path.isdir(nested_dir):
        print("Adjusting dataset directory structure...")
        for item in os.listdir(nested_dir):
            shutil.move(os.path.join(nested_dir, item), dataset_dir)
        os.rmdir(nested_dir)
        
    print("Dataset prepared successfully.")
else:
    print("Dataset already exists.")

## 4. Train Model

Load the selected YOLO26 model and start training.

In [ ]:
import torch
from ultralytics import YOLO

# Ensure W&B is disabled to run offline smoothly
os.environ['WANDB_DISABLED'] = 'true'

# Check device
device = 0 if torch.cuda.is_available() else 'cpu'
print(f"Torch version: {torch.__version__} | CUDA available: {torch.cuda.is_available()} | Using device: {device}")

# Load a pretrained YOLO26 segment model
print(f"Loading {MODEL_NAME}...")
model = YOLO(MODEL_NAME)

# Start training
print("Starting training...")
results = model.train(
    data=DATA_CONFIG,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    project=PROJECT_NAME,
    deterministic=False,
    plots=True,
    device=device,
)
print("Training completed.")

## 5. Model Validation

Evaluate the trained model's performance on the validation dataset.

In [ ]:
# Find best weights
best_weights = f"{PROJECT_NAME}/train/weights/best.pt"

if os.path.exists(best_weights):
    weights_path = best_weights
    print(f"Using trained weights: {weights_path}")
else:
    weights_path = MODEL_NAME
    print(f"Trained weights not found. Using pretrained: {weights_path}")

# Load model for validation
model_val = YOLO(weights_path)

# Run validation
metrics = model_val.val(
    data=DATA_CONFIG,
    project=PROJECT_NAME,
    imgsz=IMAGE_SIZE,
    batch=1,
    iou=IOU_THRESHOLD,
    conf=CONF_THRESHOLD,
    half=True,
    save_json=True,
    save_txt=True,
    save_conf=True,
    plots=True,
    device=device,
)

print("Validation metrics:")
print(f"Box mAP50-95: {metrics.box.map:.4f}")
print(f"Box mAP50: {metrics.box.map50:.4f}")
print(f"Mask mAP50-95: {metrics.seg.map:.4f}")
print(f"Mask mAP50: {metrics.seg.map50:.4f}")

## 6. Inference and Visualization

Perform inference on test images and display the predictions.

In [ ]:
import glob
import cv2
import matplotlib.pyplot as plt

# Run inference on validation/test images to see quality
test_images_dir = "./AlphaDent/images/valid/"
output_dir = "./inference_output/"

model_infer = YOLO(weights_path)

print(f"Running inference on images from: {test_images_dir}")
results_infer = model_infer.predict(
    source=test_images_dir,
    project=output_dir,
    name='preds',
    imgsz=IMAGE_SIZE,
    conf=0.25,  # higher threshold for visual clarity
    iou=0.6,
    half=True,
    save=True,
    save_txt=True,
    device=device
)

# Plot the first few results
saved_images = glob.glob(f"{output_dir}/preds/*.jpg") + glob.glob(f"{output_dir}/preds/*.png")
if saved_images:
    num_display = min(3, len(saved_images))
    fig, axes = plt.subplots(1, num_display, figsize=(18, 6))
    if num_display == 1:
        axes = [axes]
    for idx, img_path in enumerate(saved_images[:num_display]):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(img)
        axes[idx].axis('off')
        axes[idx].set_title(os.path.basename(img_path))
    plt.tight_layout()
    plt.show()
else:
    print("No output images found to display.")